# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 claim: "A page that hasn't been updated in a long time (days_since_last_update) is more likely to be in a declining trend."

Signal 2 claim: "A page ranking well (avg_position) but with low CTR relative to other pages at similar position is a sign of a CTR problem, not a content problem."

Say which one is flag-linked and to which flag (staleness → stale_visible_page/refresh flags; CTR-vs-position → needs_ctr_fix).

**Signal** 1 (staleness, flag-linked to stale_visible_page): **CONFIRMED**

**Signal** 2 (CTR-vs-position, flag-linked to needs_ctr_fix): **MIXED**

**Rule:** Flag pages that have not been updated recently but still have meaningful search visibility, because stale pages with continued visibility are stronger candidates for a content refresh.

**Reason code:** `stale_visible_page` — a page flagged because it is both stale (`days_since_last_update >= 90`) and still visible (`impressions_90d >= 300`).


Signal 1: **staleness**

In [20]:
!git clone https://github.com/aleezafatima-21/Aleeza-flyrank-ml-internship.git /content/Aleeza-flyrank-ml-internship

fatal: destination path '/content/Aleeza-flyrank-ml-internship' already exists and is not an empty directory.


In [21]:
import pandas as pd, numpy as np

csv_path = "/content/Aleeza-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

bins = [0, 30, 90, 180, 365, 10000]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=bins, labels=labels,
    right=True, include_lowest=True
)

signal1_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean")
)
print("Overall decline rate:", df["is_declining_label"].mean())
print(signal1_table)

Overall decline rate: 0.5420666666666667
                      n  decline_rate
staleness_bucket                     
0-30              20480      0.511377
31-90               175      0.588571
91-180             9171      0.611057
181-365             169      0.467456
365+                  5      0.600000


Signal 2: CTR vs **position**

In [22]:
sub = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()

pos_bins = [0, 10, 20, 50, 1000]
pos_labels = ["top10", "11-20", "21-50", "50+"]
sub["pos_bucket"] = pd.cut(sub["avg_position"], bins=pos_bins, labels=pos_labels, right=True)

signal2_table = sub.groupby("pos_bucket", observed=True).agg(
    n=("ctr", "size"),
    median_ctr=("ctr", "median"),
    decline_rate=("is_declining_label", "mean")
)
print(signal2_table)

               n  median_ctr  decline_rate
pos_bucket                                
top10       9215        0.23      0.615084
11-20       5876        0.15      0.626106
21-50       6037        0.06      0.584065
50+          878        0.00      0.317768


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
import os

# --- the rule, as two conditions ---
stale   = (df["days_since_last_update"] >= 90).astype(int)    # matches elevated decline rate seen in Section 1 (91-180 bucket)
visible = (df["impressions_90d"] >= 300).astype(int)           # "moderate" visibility per data dictionary's impression_tier

df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = "stale_visible_page"
df["action"] = df["baseline_score"].apply(lambda s: "refresh_review" if s > 0 else "monitor")

ranked_queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "days_since_last_update", "impressions_90d",
     "avg_position", "ctr", "baseline_score", "reason_code", "action"]
]

os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows flagged for review:", (df['action'] == 'refresh_review').sum())
ranked_queue.head(10)

Rows flagged for review: 7234


,content_id,days_since_last_update,impressions_90d,avg_position,ctr,baseline_score,reason_code,action
6653,content_5fe46e04994d,104,517715,4.2,0.14,517715,stale_visible_page,refresh_review
29400,content_2dba2b1f9536,104,443434,27.9,0.21,443434,stale_visible_page,refresh_review
13537,content_2c2606c5d176,104,347399,4.2,0.53,347399,stale_visible_page,refresh_review
26531,content_cb112fce36be,104,309910,5.6,0.16,309910,stale_visible_page,refresh_review
21565,content_9532f197bbc8,104,309192,2.0,0.87,309192,stale_visible_page,refresh_review
3394,content_36ff89c8214e,104,295097,7.3,0.05,295097,stale_visible_page,refresh_review
26798,content_b28d1efd668f,104,286608,26.2,0.06,286608,stale_visible_page,refresh_review
23767,content_813e88069237,104,233561,26.2,0.06,233561,stale_visible_page,refresh_review
26255,content_c21024970297,104,211366,5.1,0.41,211366,stale_visible_page,refresh_review
7445,content_c8e9d6ab9013,104,208678,9.7,0.00,208678,stale_visible_page,refresh_review


In [24]:
!ls -la work/outputs/

total 2028
drwxr-xr-x 2 root root    4096 Aug 27 11:43 .
drwxr-xr-x 3 root root    4096 Aug 27 11:43 ..
-rw-r--r-- 1 root root 2065817 Aug 27 12:07 baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
top10_review = ranked_queue.head(10)[
    ["content_id", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "action"]
]
top10_review

,content_id,days_since_last_update,impressions_90d,avg_position,ctr,action
6653,content_5fe46e04994d,104,517715,4.2,0.14,refresh_review
29400,content_2dba2b1f9536,104,443434,27.9,0.21,refresh_review
13537,content_2c2606c5d176,104,347399,4.2,0.53,refresh_review
26531,content_cb112fce36be,104,309910,5.6,0.16,refresh_review
21565,content_9532f197bbc8,104,309192,2.0,0.87,refresh_review
3394,content_36ff89c8214e,104,295097,7.3,0.05,refresh_review
26798,content_b28d1efd668f,104,286608,26.2,0.06,refresh_review
23767,content_813e88069237,104,233561,26.2,0.06,refresh_review
26255,content_c21024970297,104,211366,5.1,0.41,refresh_review
7445,content_c8e9d6ab9013,104,208678,9.7,0.00,refresh_review


**1.** **`content_5fe46e04994d`** — Action: `refresh_review`. Why: 517,715 impressions/90d, last updated 104 days ago, avg_position 4.2, and CTR 0.14, so it has strong visibility but is stale. What would make it wrong: if the page is already stable or improving despite being stale, a refresh may not be necessary. Also, its high position in the queue is influenced heavily by its large `impressions_90d`; because all these pages have the same 104-day staleness value, the ranking is largely putting the biggest-traffic pages first rather than distinguishing how stale they are.

**2.** **`content_2dba2b1f9536`** — Action: `refresh_review`. Why: 443,434 impressions/90d and 104 days since the last update show strong visibility with meaningful staleness, although avg_position 27.9 means it is not ranking near the top. What would make it wrong: if the page's low ranking is caused by a problem that a content refresh cannot address.

**3.** **`content_2c2606c5d176`** — Action: `refresh_review`. Why: 347,399 impressions/90d, 104 days since update, avg_position 4.2, and CTR 0.53 indicate a highly visible page that still ranks well but is stale. What would make it wrong: if its performance is stable or improving without an update.

**4.** **`content_cb112fce36be`** — Action: `refresh_review`. Why: 309,910 impressions/90d and 104 days since update show that this is a high-visibility page that has become stale; its avg_position is 5.6. What would make it wrong: if the page is maintaining or improving its rankings and traffic despite being stale.

**5.** **`content_9532f197bbc8`** — Action: `refresh_review`. Why: 309,192 impressions/90d, 104 days since update, and avg_position 2.0 show very strong visibility and ranking, while the page is still stale. What would make it wrong: if its strong performance is stable and there is no evidence that updating it would improve results. Also, its queue position is strongly influenced by its high `impressions_90d`; since the top pages share the same 104-day staleness value, the ranking reflects traffic size as much as the staleness signal.

**6.** **`content_36ff89c8214e`** — Action: `refresh_review`. Why: 295,097 impressions/90d and 104 days since update make it a highly visible stale page; avg_position 7.3 is strong, but CTR is only 0.05. What would make it wrong: Signal 2 was MIXED, so the low CTR alone should not be treated as proof that a refresh will help.

**7.** **`content_b28d1efd668f`** — Action: `refresh_review`. Why: 286,608 impressions/90d and 104 days since update give it strong visibility and staleness, but avg_position 26.2 shows that it is not currently ranking very well. What would make it wrong: if the poor position reflects a broader ranking problem that cannot reasonably be fixed through a content refresh.

**8.** **`content_813e88069237`** — Action: `refresh_review`. Why: 233,561 impressions/90d and 104 days since update indicate a stale page with substantial visibility, although its avg_position of 26.2 is relatively weak. What would make it wrong: if the page's traffic is high mainly because of broad impressions while its actual search performance is declining for reasons unrelated to freshness.

**9.** **`content_c21024970297`** — Action: `refresh_review`. Why: 211,366 impressions/90d, 104 days since update, avg_position 5.1, and CTR 0.41 indicate a stale page with strong visibility and good ranking. What would make it wrong: if the page is already stable or improving, making the refresh unnecessary.

**10.** **`content_c8e9d6ab9013`** — Action: `refresh_review`. Why: 208,678 impressions/90d and 104 days since update show strong visibility and staleness; avg_position is 9.7, but CTR is 0.00. What would make it wrong: because Signal 2 was MIXED, the very low CTR should not by itself justify a refresh; if the page is otherwise stable, the recommendation could be wrong.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Part (a) — Weak picks

The weakest picks are `content_c8e9d6ab9013`, `content_b28d1efd668f`, and `content_813e88069237`. The first has a CTR of 0.00, but Signal 2 was MIXED, so low CTR alone is not strong evidence for a refresh. The other two have avg_position 26.2, so their high impressions do not necessarily mean they are strong refresh opportunities.

A broader weakness is that the queue is dominated by pages with the same `days_since_last_update` value of 104, while `impressions_90d` drives the ranking among them. The rule does not distinguish different degrees of staleness well: a borderline-stale page with very high traffic can rank ahead mainly because of its traffic volume rather than because it is substantially more stale.

### Part (b) — Leakage check

I did not use `trend_direction` or `trend_pct` in `baseline_score`, `reason_code`, or `action`.

I did not use any product flag columns (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, or `is_quick_win`). These columns are not available in this dataset/CSV.

I did not use future-window data to peek ahead. The rule uses `days_since_last_update` and `impressions_90d`, which describe the page's current/historical state rather than a future performance window.


In [26]:
leaky_terms = ["trend_direction", "trend_pct", "health_score", "priority_score",
               "action_type", "needs_ctr_fix", "is_quick_win"]
used_columns = ["days_since_last_update", "impressions_90d"]

print("Columns used in the rule:", used_columns)
print("Any leaky terms among them?", any(term in used_columns for term in leaky_terms))
print("Leaky terms present in dataset at all?", [t for t in leaky_terms if t in df.columns])

Columns used in the rule: ['days_since_last_update', 'impressions_90d']
Any leaky terms among them? False
Leaky terms present in dataset at all? ['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.